# Introduction to Distillation via PyTorch

### Setup a Teacher and Student model

In [12]:
### Overall goal is to rebuild the CIFAR-10 classifier with a smaller model ###
###
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# Check if the current `accelerator <https://pytorch.org/docs/stable/torch.html#accelerators>`__
# is available, and if not, use the CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [13]:
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

#loading the CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

In [14]:
#For CPU users who want a small scale experiment.

#from torch.utils.data import Subset
#num_images_to_keep = 2000
#train_dataset = Subset(train_dataset, range(min(num_images_to_keep, 50_000)))
#test_dataset = Subset(test_dataset, range(min(num_images_to_keep, 10_000)))

In [15]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

In [16]:
#Deep NN used as a taecher
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1), #wtf is dropout??,
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x) #Pass through conv layers
        x = torch.flatten(x, 1) # flatten everything 
        x = self.classifier(x) #Pass through linear layers
        return x
    
#Light NN used as a student
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x 
    

In [17]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()                                   #Loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)        #Change parameters through Adam
    
    model.train()
    
    for epoch in range(epochs):
        running_loss= 0.0
        for inputs, labels in train_loader:
            #inputs: collection of batch-sized images
            #labels: vector of dimensionality batch_size w integers denoting class of each image
            inputs, labels = inputs.to(device), labels.to(device) #to(device) moves data to GPU if avail
            
            optimizer.zero_grad() #clear grad of prev batch
            outputs = model(inputs) #run forward pass(call forward) -> output.shape = (batch_size, num_classes)
            
            #outputs: output of the network. Tensor of dimension batch_size x num_classes
            #labels: actual labels of image. Vector of dimension batch_size
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step() #BACKPROPAGATION yipeee, updating the weights
            
            running_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")
def test(model, test_loader, device):
    model.to(device)
    model.eval()                                                            #Disable dropout, changes batch_norm behaviour
    
    correct = 0
    total = 0
    
    with torch.no_grad():                                                   #Don't track gradients         
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    accuracy = 100 * correct / total
    print(f"Test accuracy: {accuracy: .2f}%")
    return accuracy

In [18]:
torch.manual_seed(53)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device=device)


torch.manual_seed(53)
nn_light = LightNN(num_classes=10).to(device)

Epoch 1/10, Loss: 1.394295476434176
Epoch 2/10, Loss: 0.9048893020281097
Epoch 3/10, Loss: 0.7060796401994612
Epoch 4/10, Loss: 0.5596853544949876
Epoch 5/10, Loss: 0.43509460417815793
Epoch 6/10, Loss: 0.32280859316858795
Epoch 7/10, Loss: 0.23827356915644674
Epoch 8/10, Loss: 0.17646243144064913
Epoch 9/10, Loss: 0.1470586004121529
Epoch 10/10, Loss: 0.12296777764511535
Test accuracy:  74.99%


In [19]:
torch.manual_seed(53)
new_nn_light = LightNN(num_classes=10).to(device)

In [20]:
# Print the norm of the first layer of the initial lightweight model
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())
# Print the norm of the first layer of the new lightweight model
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())

total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN parameters: {total_params_deep}")
total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN parameters: {total_params_light}")


Norm of 1st layer of nn_light: 2.2939505577087402
Norm of 1st layer of new_nn_light: 2.2939505577087402
DeepNN parameters: 1,186,986
LightNN parameters: 267,738


In [21]:
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.4817569158266268
Epoch 2/10, Loss: 1.1339906967814317
Epoch 3/10, Loss: 1.0010924011545108
Epoch 4/10, Loss: 0.9086360972555702
Epoch 5/10, Loss: 0.832387816875487
Epoch 6/10, Loss: 0.7664558872237535
Epoch 7/10, Loss: 0.703986368978115
Epoch 8/10, Loss: 0.6472389542538187
Epoch 9/10, Loss: 0.6019441149271357
Epoch 10/10, Loss: 0.5477427333364706
Test accuracy:  69.64%


In [22]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light_ce:.2f}%")

Teacher accuracy: 74.99%
Student accuracy: 69.64%


### Knowledge Distillation Training

In [29]:
### New parameters
# T: temperature, smoothness of output distributions. Larger means smoother distributions
# soft_target_loss_weight: Weight assigned to extra objective
# ce_loss_weight: weight assigned to cross-entropy
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    
    teacher.eval()
    student.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            #forward pass w teacher model
            with torch.no_grad():
                teacher_logits = teacher(inputs)
            
            #forward pass w student model 
            student_logits = student(inputs)
            
            #soften the student logit
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=1)
            
            # Loss calculations (no idea how they got this)
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)
            
            label_loss = ce_loss(student_logits, labels)
            
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
            #
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

# Apply ``train_knowledge_distillation`` with a temperature of 2. Arbitrarily set the weights to 0.75 for CE and 0.25 for distillation loss.
train_knowledge_distillation(teacher=nn_deep, student=new_nn_light, train_loader=train_loader, epochs=10, learning_rate=0.001, T=10, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

# Compare the student test accuracy with and without the teacher, after distillation
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")
            

Epoch 1/10, Loss: 1.7009716024789054
Epoch 2/10, Loss: 1.654139244647892
Epoch 3/10, Loss: 1.6122655459986928
Epoch 4/10, Loss: 1.5844839738153131
Epoch 5/10, Loss: 1.570689254709522
Epoch 6/10, Loss: 1.5382289176097002
Epoch 7/10, Loss: 1.5139312948412298
Epoch 8/10, Loss: 1.4995233832722734
Epoch 9/10, Loss: 1.4824999400112024
Epoch 10/10, Loss: 1.4658423982312918
Test accuracy:  71.38%
Teacher accuracy: 74.99%
Student accuracy without teacher: 69.64%
Student accuracy with CE + KD: 71.38%


In [ ]:
# torch.manual_seed(53)
# T = 2, ce = 0.75, loss_weight = 0.25 
# Test accuracy:  69.86%
# Teacher accuracy: 74.99%
# Student accuracy without teacher: 69.64%
# Student accuracy with CE + KD: 69.86%
# loss = 0.884

# T = 5, ce = 0.75, loss_weight = 0.25 
# Test accuracy:  71.23%
# Teacher accuracy: 74.99%
# Student accuracy without teacher: 69.64%
# Student accuracy with CE + KD: 71.23%
# loss = 1.147

# T = 5, ce = 3, loss_weight = 0.25
# Test accuracy:  70.46%
# Teacher accuracy: 74.99%
# Student accuracy without teacher: 69.64%
# Student accuracy with CE + KD: 70.46%

# T = 5, ce = 0.75, loss_weight = 0.5
# Test accuracy:  71.02%
# Teacher accuracy: 74.99%
# Student accuracy without teacher: 69.64%
# Student accuracy with CE + KD: 71.02%

# T = 10, ce = 0.75, loss_weight = 0.25
# Test accuracy:  71.38%
# Teacher accuracy: 74.99%
# Student accuracy without teacher: 69.64%
# Student accuracy with CE + KD: 71.38%


### Optimizing through Cosine Loss Minimization Run

In [30]:
class ModifiedDeepNNCosine(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedDeepNNCosine, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1), #wtf is dropout??,
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)
        return x, flattened_conv_output_after_pooling
    
class ModifiedLightNNCosine(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedLightNNCosine, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        flattened_conv_output = torch.flatten(x, 1)
        x = self.classifier(flattened_conv_output)
        return x, flattened_conv_output
        

In [31]:
#Apparently this loads the trained instance (weights) from the deepNN model?
modified_nn_deep = ModifiedDeepNNCosine(num_classes=10).to(device)
modified_nn_deep.load_state_dict(nn_deep.state_dict())

<All keys matched successfully>

In [32]:
print("Norm of 1st layer for deep_nn:", torch.norm(nn_deep.features[0].weight).item())
print("Norm of 1st layer for modified_deep_nn:", torch.norm(modified_nn_deep.features[0].weight).item())

Norm of 1st layer for deep_nn: 7.681605339050293
Norm of 1st layer for modified_deep_nn: 7.681605339050293


In [33]:
torch.manual_seed(42)
modified_nn_light = ModifiedLightNNCosine(num_classes=10).to(device)
print("Norm of 1st layer:", torch.norm(modified_nn_light.features[0].weight).item())

Norm of 1st layer: 2.327361822128296


In [34]:
sample_input = torch.randn(128,3 , 32, 32).to(device) #batch_size = 128, filters = 3, image = 32x32

# Pass the input through the student
logits, hidden_representation = modified_nn_light(sample_input)

# Print the shapes of the tensors
print("Student logits shape:", logits.shape) # batch_size x total_classes
print("Student hidden representation shape:", hidden_representation.shape) # batch_size x hidden_representation_size

# Pass the input through the teacher
logits, hidden_representation = modified_nn_deep(sample_input)

# Print the shapes of the tensors
print("Teacher logits shape:", logits.shape) # batch_size x total_classes
print("Teacher hidden representation shape:", hidden_representation.shape) # batch_size x hidden_representation_size

Student logits shape: torch.Size([128, 10])
Student hidden representation shape: torch.Size([128, 1024])
Teacher logits shape: torch.Size([128, 10])
Teacher hidden representation shape: torch.Size([128, 1024])


In [41]:
def train_cosine_loss(teacher, student, train_loader, epochs, learning_rate, hidden_rep_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    cosine_loss = nn.CosineEmbeddingLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)
    
    teacher.to(device)
    student.to(device)
    teacher.eval()
    student.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            with torch.no_grad():
                _, teacher_hidden_representation = teacher(inputs)
                
            student_logits, student_hidden_representation = student(inputs)
            
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(device))
            
            label_loss = ce_loss(student_logits, labels)
            
            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}") 
                

In [42]:
def test_multiple_outputs(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs, _ = model(inputs) # Disregard the second tensor of the tuple
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [ ]:
# Train and test the lightweight network with cross entropy loss
train_cosine_loss(teacher=modified_nn_deep, student=modified_nn_light, train_loader=train_loader, epochs=10, learning_rate=0.001, hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, test_loader, device)

Epoch 1/10, Loss: 1.295226437356466
Epoch 2/10, Loss: 1.0652494695790284
Epoch 3/10, Loss: 0.9653890393579098
Epoch 4/10, Loss: 0.898505659511937
Epoch 5/10, Loss: 0.8419842960889382
Epoch 6/10, Loss: 0.7988138055557485
Epoch 7/10, Loss: 0.7516005767885682
Epoch 8/10, Loss: 0.7155065559365256
Epoch 9/10, Loss: 0.6835095123256869
Epoch 10/10, Loss: 0.6532662044400754
Test Accuracy: 70.14%


: 